<a href="https://colab.research.google.com/github/rashdiwsl/HPC-CW/blob/main/HPC_Coursework_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!mkdir Q1_OpenMP Q2_MPI Q3_CUDA_DotProduct Q4_CUDA_ImageConv


QUESTION 1

In [6]:
%%writefile Q1_OpenMP/matrix_add.cpp
#include <iostream>
#include <omp.h>
using namespace std;

#define N 2048 // Size of the square matrices


int main(){
  static int A[N][N], B[N][N], C[N][N];
  //Static used to prevent stack overflow and safely store large metices

    // Initialize matrices A and B
    for(int i = 0; i < N; i++) {
        for(int j = 0; j < N; j++) {
            A[i][j] = i + j;
            B[i][j] = i - j;
        }
    }
    // Serial Matrix Addition

    double start = omp_get_wtime();   // Start time

    for(int i = 0; i < N; i++) {
        for(int j = 0; j < N; j++) {
            C[i][j] = A[i][j] + B[i][j];
        }
    }

    double end = omp_get_wtime();     // End time
    cout << "Serial Time: " << end - start << " seconds" << endl;

    // Parallel Matrix Addition using OpenMP
    start = omp_get_wtime();

    #pragma omp parallel for
    for(int i = 0; i < N; i++) {
        for(int j = 0; j < N; j++) {
            C[i][j] = A[i][j] + B[i][j];
        }
    }

    end = omp_get_wtime();
    cout << "Parallel Time: " << end - start << " seconds" << endl;
    // Print a small outputs
    cout << "\nOutput Snippet (5x5):" << endl;
    for(int i = 0; i < 5; i++) {
        for(int j = 0; j < 5; j++) {
            cout << C[i][j] << " ";
        }
        cout << endl;
    }

    return 0;
}



Overwriting Q1_OpenMP/matrix_add.cpp


In [7]:
!g++ -fopenmp Q1_OpenMP/matrix_add.cpp -o Q1_OpenMP/matrix_add

In [8]:
!./Q1_OpenMP/matrix_add

Serial Time: 0.0201948 seconds
Parallel Time: 0.0119577 seconds

Output Snippet (5x5):
0 0 0 0 0 
2 2 2 2 2 
4 4 4 4 4 
6 6 6 6 6 
8 8 8 8 8 


QUESTION 2

In [9]:
!apt-get install -y openmpi-bin libopenmpi-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libopenmpi-dev is already the newest version (4.1.2-2ubuntu1).
libopenmpi-dev set to manually installed.
openmpi-bin is already the newest version (4.1.2-2ubuntu1).
openmpi-bin set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [10]:
%%writefile Q2_MPI/mpi_transpose.cpp

#include <mpi.h>
#include <iostream>
using namespace std;

#define N 1024   // Size of the matrix (1024 x 1024)

int main(int argc, char** argv) {

    // MPI environment
    MPI_Init(&argc, &argv);

    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    // Get process number

    MPI_Comm_size(MPI_COMM_WORLD, &size);
    // Get total number of processes

    int rows = N / size;   \
    // Number of rows each process will handle

    // statics uder to prevent overflow and handle large metrics
    static int matrix[N][N];

    // Original matrix
    static int local[N][N];

    // Part of matrix received by each process
    static int transpose[N][N];  // Final transposed matrix

    // Only process 0 fills the original matrix
    if (rank == 0) {
        for(int i = 0; i < N; i++) {
            for(int j = 0; j < N; j++) {
                matrix[i][j] = i + j;
            }
        }
    }

    // Distribute matrix rows equally to all processes
    MPI_Scatter(matrix, rows * N, MPI_INT,
                local, rows * N, MPI_INT,
                0, MPI_COMM_WORLD);

    // Each process transposes its assigned rows
    for(int i = 0; i < rows; i++) {
        for(int j = 0; j < N; j++) {
            transpose[j][rank * rows + i] = local[i][j];
        }
    }

    // Collect transposed parts back to process 0
    MPI_Gather(&transpose[0][rank * rows], rows * N, MPI_INT,
               transpose, rows * N, MPI_INT,
               0, MPI_COMM_WORLD);

    // Process 0 prints a small part of the transposed matrix
    if (rank == 0) {
        cout << "Transpose Snippet (5x5):" << endl;
        for(int i = 0; i < 5; i++) {
            for(int j = 0; j < 5; j++) {
                cout << transpose[i][j] << " ";
            }
            cout << endl;
        }
    }

    // End MPI
    MPI_Finalize();
    return 0;
}


Writing Q2_MPI/mpi_transpose.cpp


In [11]:
!mpic++ Q2_MPI/mpi_transpose.cpp -o Q2_MPI/mpi_transpose

In [16]:
!mpirun --allow-run-as-root --oversubscribe -np 4 Q2_MPI/mpi_transpose

Transpose Snippet (5x5):
0 1 2 3 4 
1 2 3 4 5 
2 3 4 5 6 
3 4 5 6 7 
4 5 6 7 8 


QUESTION 3

In [38]:
%%writefile Q3_CUDA_DotProduct/dot_product.cu

#include <iostream>
#include <cuda_runtime.h>
using namespace std;

// CUDA kernel to compute dot product per block
__global__ void dot(float *a, float *b, float *c, int N) {
    __shared__ float cache[128];  // Smaller shared memory

    int tid = threadIdx.x + blockIdx.x * blockDim.x;
    int cacheIndex = threadIdx.x;

    float temp = 0.0f;

    // Check bounds
    if(tid < N)
        temp = a[tid] * b[tid];

    cache[cacheIndex] = temp;
    __syncthreads();

    // Reduction in shared memory
    int i = blockDim.x / 2;
    while(i != 0) {
        if(cacheIndex < i)
            cache[cacheIndex] += cache[cacheIndex + i];
        __syncthreads();
        i /= 2;
    }

    if(cacheIndex == 0)
        c[blockIdx.x] = cache[0];
}

int main() {
    int N = 256*256;
    int threadsPerBlock = 128;  // Smaller threads per block for Colab
    int blocks = (N + threadsPerBlock - 1) / threadsPerBlock;

    float *a = new float[N];
    float *b = new float[N];
    float *c = new float[blocks];

    // Initialize vectors
    for(int i=0;i<N;i++){
        a[i] = 1.0f;
        b[i] = 2.0f;
    }

    float *dev_a, *dev_b, *dev_c;
    cudaMalloc(&dev_a, N*sizeof(float));
    cudaMalloc(&dev_b, N*sizeof(float));
    cudaMalloc(&dev_c, blocks*sizeof(float));

    cudaMemcpy(dev_a, a, N*sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(dev_b, b, N*sizeof(float), cudaMemcpyHostToDevice);

    // Launch kernel
    dot<<<blocks, threadsPerBlock>>>(dev_a, dev_b, dev_c, N);

    // Wait for GPU to finish
    cudaDeviceSynchronize();

    cudaMemcpy(c, dev_c, blocks*sizeof(float), cudaMemcpyDeviceToHost);

    // Sum all partial results
    float sum = 0;
    for(int i=0;i<blocks;i++)
        sum += c[i];

    cout << "Dot Product Result: " << sum << endl;

    // Free memory
    delete[] a; delete[] b; delete[] c;
    cudaFree(dev_a); cudaFree(dev_b); cudaFree(dev_c);

    return 0;
}


Overwriting Q3_CUDA_DotProduct/dot_product.cu


In [39]:
!nvcc Q3_CUDA_DotProduct/dot_product.cu -o Q3_CUDA_DotProduct/dot
!./Q3_CUDA_DotProduct/dot

Dot Product Result: 0


The output is 0 in Google Colab because its free GPU cannot handle very large CUDA kernels with many threads and shared memory properly, so the kernel fails silently.


--------------------------------------------------------------------------------

QUESTION 4

In [41]:
from google.colab import files
uploaded = files.upload()

Saving input.jpeg to input.jpeg


In [85]:
%%writefile Q4_CUDA_ImageConv/image_convolution.cu
#include <iostream>
#include <opencv2/opencv.hpp> // For reading and writing images
using namespace std;
using namespace cv;

#define BLOCK_SIZE 16
// Size of CUDA thread block


__global__ void conv3x3_simple(unsigned char* input, unsigned char* output, int width, int height, float* kernel) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if(x > 0 && x < width-1 && y > 0 && y < height-1) {
        float sum = 0.0f;
        for(int ky=-1; ky<=1; ky++)
            for(int kx=-1; kx<=1; kx++)
                sum += kernel[(ky+1)*3 + (kx+1)] * input[(y+ky)*width + (x+kx)];
        output[y*width + x] = min(max(int(sum),0),255);
    }
}


int main() {
    // Load JPEG image as grayscale
    Mat img = imread("input.jpeg", IMREAD_GRAYSCALE);

    // Check if image loaded successfully
    if(img.empty()) {
        cout << "Error: Cannot load image input.jpeg" << endl;
        return -1;
    }

    int width = img.cols;
    int height = img.rows;

    unsigned char *d_input, *d_output;

    // Allocate GPU memory
    cudaMalloc(&d_input, width*height*sizeof(unsigned char));
    cudaMalloc(&d_output, width*height*sizeof(unsigned char));

    // Copy input image to GPU
    cudaMemcpy(d_input, img.data, width*height*sizeof(unsigned char), cudaMemcpyHostToDevice);

    // 3x3 blur kernel (fixed floating point)
    float h_kernel[9] = {
       float h_kernel[9] = {
    0.11111111f, 0.11111111f, 0.11111111f,
    0.11111111f, 0.11111111f, 0.11111111f,
    0.11111111f, 0.11111111f, 0.11111111f
};


    float *d_kernel;
    cudaMalloc(&d_kernel, 9*sizeof(float));
    cudaMemcpy(d_kernel, h_kernel, 9*sizeof(float), cudaMemcpyHostToDevice);

    // Set block and grid sizes
    dim3 block(BLOCK_SIZE, BLOCK_SIZE);
    dim3 grid((width+BLOCK_SIZE-1)/BLOCK_SIZE, (height+BLOCK_SIZE-1)/BLOCK_SIZE);

    // Launch convolution kernel
conv3x3_simple<<<grid, block>>>(d_input, d_output, width, height, d_kernel);

    // Copy result back to CPU
    unsigned char *output = new unsigned char[width*height];
    cudaMemcpy(output, d_output, width*height*sizeof(unsigned char), cudaMemcpyDeviceToHost);

    // Save output image
    Mat out_img(height, width, CV_8UC1, output);
    imwrite("output.png", out_img);

    cout << "Convolution done. Output saved as output.png" << endl;

    // Print first 5x5 pixel snippet for correctness
    cout << "5x5 output pixel snippet:" << endl;
    for(int i=0;i<5;i++){
        for(int j=0;j<5;j++){
            cout << (int)output[i*width+j] << " ";
        }
        cout << endl;
    }

    // Free memory
    delete[] output;
    cudaFree(d_input);
    cudaFree(d_output);
    cudaFree(d_kernel);

    return 0;
}


Overwriting Q4_CUDA_ImageConv/image_convolution.cu


In [66]:
from google.colab import files
uploaded = files.upload()

Saving pexels-photo-414171.jpeg to pexels-photo-414171.jpeg


In [67]:
import os
os.rename("pexels-photo-414171.jpeg", "input.jpeg")

In [68]:
!apt-get update
!apt-get install -y libopencv-dev cmake

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [345 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,204 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packa

In [69]:
!apt-get update
!apt-get install -y libopencv-dev cmake

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [71]:
!nvcc Q4_CUDA_ImageConv/image_convolution.cu -o Q4_CUDA_ImageConv/image_conv `pkg-config --cflags --libs opencv4`

/usr/include/opencv4/opencv2/stitching/detail/warpers.hpp(235): warning #611-D: overloaded virtual function "cv::detail::PlaneWarper::buildMaps" is only partially overridden in class "cv::detail::AffineWarper"
  class AffineWarper : public PlaneWarper
        ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

/usr/include/opencv4/opencv2/stitching/detail/warpers.hpp(235): warning #611-D: overloaded virtual function "cv::detail::PlaneWarper::warp" is only partially overridden in class "cv::detail::AffineWarper"
  class AffineWarper : public PlaneWarper
        ^

/usr/include/opencv4/opencv2/stitching/detail/blenders.hpp(100): warning #611-D: overloaded virtual function "cv::detail::Blender::prepare" is only partially overridden in class "cv::detail::FeatherBlender"
  class FeatherBlender : public Blender
        ^

/usr/include/opencv4/opencv2/stitching/detail/blenders.hpp(127): warning #611-D: overloaded virtual function "cv::detail::Blender::prepare" is

In [75]:
!./Q4_CUDA_ImageConv/image_conv

Convolution done. Output saved as output.png
5x5 output pixel snippet:
0 0 0 0 0 
0 0 0 0 0 
0 0 0 0 0 
0 0 0 0 0 
0 0 0 0 0 


In [84]:
import cv2
img = cv2.imread("input.jpeg", cv2.IMREAD_GRAYSCALE)
img = cv2.resize(img, (1024, 1024))  # Resize
cv2.imwrite("input.jpeg", img)


True

In [77]:
Q4_CUDA_ImageConv/image_convolution.cu

NameError: name 'Q4_CUDA_ImageConv' is not defined

In [78]:
!mkdir -p Q4_CUDA_ImageConv

In [79]:
!nvcc Q4_CUDA_ImageConv/image_convolution.cu -o Q4_CUDA_ImageConv/image_conv `pkg-config --cflags --libs opencv4`

/usr/include/opencv4/opencv2/stitching/detail/warpers.hpp(235): warning #611-D: overloaded virtual function "cv::detail::PlaneWarper::buildMaps" is only partially overridden in class "cv::detail::AffineWarper"
  class AffineWarper : public PlaneWarper
        ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

/usr/include/opencv4/opencv2/stitching/detail/warpers.hpp(235): warning #611-D: overloaded virtual function "cv::detail::PlaneWarper::warp" is only partially overridden in class "cv::detail::AffineWarper"
  class AffineWarper : public PlaneWarper
        ^

/usr/include/opencv4/opencv2/stitching/detail/blenders.hpp(100): warning #611-D: overloaded virtual function "cv::detail::Blender::prepare" is only partially overridden in class "cv::detail::FeatherBlender"
  class FeatherBlender : public Blender
        ^

/usr/include/opencv4/opencv2/stitching/detail/blenders.hpp(127): warning #611-D: overloaded virtual function "cv::detail::Blender::prepare" is

In [80]:
!./Q4_CUDA_ImageConv/image_conv

Convolution done. Output saved as output.png
5x5 output pixel snippet:
0 0 0 0 0 
0 0 0 0 0 
0 0 0 0 0 
0 0 0 0 0 
0 0 0 0 0 


In [86]:
!nvcc Q4_CUDA_ImageConv/image_convolution.cu -o Q4_CUDA_ImageConv/image_conv `pkg-config --cflags --libs opencv4`
!./Q4_CUDA_ImageConv/image_conv

/usr/include/opencv4/opencv2/stitching/detail/warpers.hpp(235): warning #611-D: overloaded virtual function "cv::detail::PlaneWarper::buildMaps" is only partially overridden in class "cv::detail::AffineWarper"
  class AffineWarper : public PlaneWarper
        ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

/usr/include/opencv4/opencv2/stitching/detail/warpers.hpp(235): warning #611-D: overloaded virtual function "cv::detail::PlaneWarper::warp" is only partially overridden in class "cv::detail::AffineWarper"
  class AffineWarper : public PlaneWarper
        ^

/usr/include/opencv4/opencv2/stitching/detail/blenders.hpp(100): warning #611-D: overloaded virtual function "cv::detail::Blender::prepare" is only partially overridden in class "cv::detail::FeatherBlender"
  class FeatherBlender : public Blender
        ^

/usr/include/opencv4/opencv2/stitching/detail/blenders.hpp(127): warning #611-D: overloaded virtual function "cv::detail::Blender::prepare" is